In [1]:
import obspy
import json

from obspy.clients.fdsn import Client

In [2]:
client = Client("USGS")

In [ ]:
catalog = client.get_events(minlatitude=42, 
                            maxlatitude=49, 
                            minlongitude=-128, 
                            maxlongitude=-118,
                            starttime=obspy.UTCDateTime("2025-01-01"), 
                            endtime=obspy.UTCDateTime("2026-01-01"))

In [5]:
features = []

for event in catalog:
    origin = event.preferred_origin()
    magnitude = event.preferred_magnitude()
    otime = origin.time
    eid = origin.resource_id.id.split("/")[3]
    
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [round(origin.longitude, 3), round(origin.latitude, 3)]
        },
        "properties": {
            "depth": origin.depth,
            "magnitude": round(magnitude.mag, 2),
            "event_id": eid,
            "event_link": f'<a href=\"https://earthquake.usgs.gov/earthquakes/eventpage/{eid}\" target=\"_blank\">https://earthquake.usgs.gov/earthquakes/eventpage/{eid}</a>',
            "origin_time": str(origin.time),
        }
    }

    features.append(feature)

geojson = {
    "type": "FeatureCollection",
    "features": features
}

with open("pnw_eq_catalog_2025.geojson", "w") as f:
    json.dump(geojson, f)